In [1]:
import os

import torch
from tianshou.algorithm import PPO
from tianshou.algorithm.modelfree.reinforce import ProbabilisticActorPolicy
from tianshou.algorithm.optim import AdamOptimizerFactory, LRSchedulerFactoryLinear
from tianshou.data import Collector, CollectStats, VectorReplayBuffer
from tianshou.trainer import OnPolicyTrainerParams
from tianshou.utils.net.common import ActorCritic, Net
from tianshou.utils.net.continuous import ContinuousActorProbabilistic, ContinuousCritic
from torch import nn


In [2]:
from utils import make_mujoco_env, export_onnx, save_fn, init_orthogonal, distribution_fn, log_tensorboard

In [3]:
task: str = "Reacher-v5"
persistence_base_dir: str = "./logs"

num_training_envs = 16
num_test_envs = 16

lr = 3e-4

epoch = 100
epoch_num_steps = 30000
collection_step_num_env_steps = 2048
buffer_size = 4096
batch_size: int = 64
k_epoch: int = 10

bound_action_method = "clip"

gamma = .99
gae_lambda = .95
max_grad_norm = .5
vf_coef = .25
ent_coef = .0
return_scaling = True
eps_clip = .2
value_clip = True
dual_clip = None
advantage_normalization = True
recompute_adv = True


seed = 666

hidden_sizes = [64, 64]
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
env, training_envs, test_envs = make_mujoco_env(
    task=task,
    num_training_envs=num_training_envs,
    num_test_envs = num_test_envs,
    obs_norm=True,
)

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=30756) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


In [5]:
state_shape = env.observation_space.shape or env.observation_space.n
action_shape = env.action_space.shape or env.action_space.n
max_action = env.action_space.high[0]

print(state_shape)
print(action_shape)
print(max_action)

(10,)
(2,)
1.0


In [6]:
net_a = Net(
        state_shape=state_shape,
        hidden_sizes=hidden_sizes,
        activation=nn.Tanh,
    )

actor = ContinuousActorProbabilistic(
    preprocess_net=net_a,
    action_shape=action_shape,
    unbounded=True,
).to(device)

net_c = Net(
    state_shape=state_shape,
    hidden_sizes=hidden_sizes,
    activation=nn.Tanh,
)

critic = ContinuousCritic(preprocess_net=net_c).to(device)
actor_critic = ActorCritic(actor, critic)

In [7]:
init_orthogonal(actor_critic=actor_critic, actor=actor)

In [8]:
optim = AdamOptimizerFactory(lr=lr)

optim.with_lr_scheduler_factory(
    LRSchedulerFactoryLinear(
        max_epochs=epoch,
        epoch_num_steps=epoch_num_steps,
        collection_step_num_env_steps=collection_step_num_env_steps,
    )
)

AdamOptimizerFactory[id=5779312320, lr_scheduler_factory=LRSchedulerFactoryLinear[num_epochs=100, epoch_num_steps=30000, collection_step_num_env_steps=2048], lr=0.0003, weight_decay=0, eps=1e-08, betas=(0.9, 0.999)]

In [9]:
policy = ProbabilisticActorPolicy(
    actor=actor,
    dist_fn=distribution_fn,
    action_scaling=True,
    action_bound_method=bound_action_method,
    action_space=env.action_space,
)

/Users/asd/Documents/dev/everything-i-reach-for/.venv/lib/python3.13/site-packages/tianshou/algorithm/modelfree/reinforce.py:152: UserWarning: action_scaling and action_bound_method are only intended to deal with unbounded model action space, but found actor model bound action space with max_action=1.0. Consider using unbounded=True option of the actor model, or set action_scaling to False and action_bound_method to None.
  warnings.warn(


In [10]:
algorithm: PPO = PPO(
    policy=policy,
    critic=critic,
    optim=optim,
    gamma=gamma,
    gae_lambda=gae_lambda,
    max_grad_norm=max_grad_norm,
    vf_coef=vf_coef,
    ent_coef=ent_coef,
    return_scaling=return_scaling,
    eps_clip=eps_clip,
    value_clip=value_clip,
    dual_clip=dual_clip,
    advantage_normalization=advantage_normalization,
    recompute_advantage=recompute_adv,
)

In [11]:
buffer = VectorReplayBuffer(buffer_size, len(training_envs))
training_collector = Collector[CollectStats](
    algorithm, training_envs, buffer, exploration_noise=True
)
test_collector = Collector[CollectStats](algorithm, test_envs)

In [12]:
logger, log_path = log_tensorboard(task, persistence_base_dir, seed)

In [13]:
output_checkopoint = os.path.join(log_path, "policy.pth")
save_best_fn = save_fn(output_checkopoint, training_envs)

In [14]:
%%capture
result = algorithm.run_training(
    OnPolicyTrainerParams(
        training_collector=training_collector,
        test_collector=test_collector,
        max_epochs=epoch,
        epoch_num_steps=epoch_num_steps,
        update_step_num_repetitions=k_epoch,
        test_step_num_episodes=num_test_envs,
        batch_size=batch_size,
        collection_step_num_env_steps=collection_step_num_env_steps,
        save_best_fn=save_best_fn,
        logger=logger,
        test_in_training=False,
    )
)

In [15]:
result

InfoStats(update_step=1500, best_score=-2.930124112497904, best_reward=-2.930124112497904, best_reward_std=1.666312565215792, train_step=3072000, train_episode=np.int64(61440), test_step=80800, test_episode=np.int64(1616), timing=TimingStats(total_time=977.1994807720184, train_time=977.1994807720184, train_time_collect=0.0, train_time_update=752.0716245174408, test_time=0.0, update_speed=3143.6774788019975))

In [16]:
test_collector.reset()
collector_stats = test_collector.collect(n_episode=num_test_envs, render=0.0)

In [17]:
%%capture
export_onnx(output_checkopoint, policy, state_shape)